# Toxic Gaming Chat Classifier - Colab Training

**By Alex You**

This notebook uses Colab as a GPU runtime while keeping the project code in Python modules.

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Mount Google Drive

Set `PROJECT_DIR` to the folder that contains this repository in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your repo is stored somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/APS360/Final_Project'
%cd $PROJECT_DIR

## 3. Install Dependencies

In [ ]:
%pip install -r requirements.txt

## 4. Verify Or Prepare Expert-Labelled Data

Download L2DTnH's `2_16000_chatlogs_english_only.csv` to `data/l2dtnh/l2dtnh_english.csv`. The preparation script normalizes the messages, removes contradictory normalized labels, and creates match-grouped train/validation/test splits.

In [ ]:
from pathlib import Path
import subprocess

import pandas as pd

raw_path = Path('data/l2dtnh/l2dtnh_english.csv')
prepared_path = Path('data/l2dtnh/l2dtnh_prepared.csv')

if not raw_path.exists():
    raise FileNotFoundError(
        'Upload 2_16000_chatlogs_english_only.csv as data/l2dtnh/l2dtnh_english.csv.'
    )

# Always rebuild so grouped split assignments and the audit match the current code.
subprocess.run(['python', 'prepare_l2dtnh.py'], check=True)

df = pd.read_csv(prepared_path)
print(df.head())
print(pd.crosstab(df['split'], df['label']))
print('Groups per split:', df.groupby('split')['group_id'].nunique().to_dict())

## 5. Context-Window + Hybrid Screen (Validation Only, Then Frozen Test)

Rebuild prepared data (includes `msg_index`), then run the context-window K screen, late-fuse with the SVM on validation, three-seed the top survivors, and open grouped test only after freeze. Do not regenerate the progress report unless mean test F1 beats the SVM baseline.

In [ ]:
# Re-prepare so msg_index / within-match order match the current prepare script.
!python prepare_l2dtnh.py
!python run_context_hybrid_experiments.py --device cuda

# Optional: keep the older single-message screen for reference (already frozen as weight_7).
# !python run_lstm_experiments.py --device cuda --screen-only


## 6. Review Frozen Comparison

In [ ]:
import json
from pathlib import Path

summary_path = Path('artifacts/context_hybrid_experiment_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Winner:', summary['winner'])
    print('Mean test F1:', summary['test_aggregate']['f1'])
    print('Baseline test F1:', summary['baseline_test']['f1'])
    print('Success against baseline:', summary['error_analysis']['success_against_baseline'])
    print('Progress report update allowed:', summary['protocol']['progress_report_update_allowed'])
else:
    summary = json.loads(Path('artifacts/experiment_summary.json').read_text(encoding='utf-8'))
    print('Frozen winner:', summary['winner'])
    print('Baseline test F1:', summary['baseline_test']['f1'])
    print('LSTM three-seed test F1:', summary['lstm_test_aggregate']['f1'])


## 7. Verify Canonical Artifacts

The repository already lives in Drive, and every run writes directly to the canonical `artifacts/` directory. No post-run copying is needed.

In [ ]:
from pathlib import Path

artifact_dir = Path(PROJECT_DIR) / 'artifacts'
required = [
    'baseline_metrics.json',
    'context_hybrid_metrics.json',
    'context_hybrid_experiment_summary.json',
    'frozen_context_hybrid_config.json',
]
legacy = [
    'lstm_metrics.json',
    'experiment_summary.json',
    'frozen_lstm_config.json',
    'best_model.pt',
]
missing = [name for name in required if not (artifact_dir / name).exists()]
if missing:
    legacy_missing = [name for name in legacy if not (artifact_dir / name).exists()]
    if legacy_missing:
        raise FileNotFoundError(
            f'Missing artifacts. context/hybrid: {missing}; legacy: {legacy_missing}'
        )
    print(
        f'Context/hybrid artifacts not ready yet ({missing}); '
        f'legacy evidence present in {artifact_dir}'
    )
else:
    print(f'Context/hybrid evidence verified in {artifact_dir}')
